In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
df_raw=pd.read_csv("discrete_tumor_id3_practice.csv")

In [2]:
df=df_raw.copy()
df.info()
print(df.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age_group    30 non-null     object
 1   tumor_size   30 non-null     object
 2   shape        30 non-null     object
 3   margin       30 non-null     object
 4   texture      30 non-null     object
 5   pain         30 non-null     object
 6   mobility     30 non-null     object
 7   lymph_nodes  30 non-null     object
 8   diagnosis    30 non-null     object
dtypes: object(9)
memory usage: 2.2+ KB
age_group      3
tumor_size     3
shape          3
margin         2
texture        3
pain           2
mobility       3
lymph_nodes    2
diagnosis      2
dtype: int64


In [3]:
x_cols=df.drop(['diagnosis'],axis=1).columns.to_list()
y_col='diagnosis'

def cal_entropy(series:pd.Series)->float:
    series=series.dropna()
    if len(series)==0:
        return 0.0
    prob=series.value_counts(normalize=True)
    entropy=-np.sum(prob*np.log2(prob))
    return float(entropy)
    

def cal_con_entropy(df:pd.DataFrame,x_col:str,y_col:str)->float:
    prob_x=df[x_col].value_counts(normalize=True)
    group_entropy=df.groupby(x_col)[y_col].apply(cal_entropy)
    return float(np.sum(prob_x*group_entropy))

class TreeNode:
    def __init__(self,is_leaf=False,feature=None,prediction=None):
        self.is_leaf=is_leaf
        self.feature=feature
        self.children={}
        self.prediction=prediction
    def display(self,indent=""):
        if(self.is_leaf):
            print(f"{indent}🍃 预测结果: {self.prediction}")
            return
        print(f"{indent}🌲 [按特征分裂: {self.feature}]")
        for val,child in self.children.items():
            print(f"{indent}  ├── 当该特征取值为 -> '{val}':")
            child.display(indent + "  │   ")

def build_tree(
    df:pd.DataFrame,
    features:list,
    target_col:str,
    entropy_threshold: float = 0.1
)->TreeNode:
    current_entropy = cal_entropy(df[target_col])
    unique_targets=df[target_col].unique()
    if len(unique_targets)==1:
        return TreeNode(is_leaf=True,prediction=unique_targets[0])
    if (current_entropy<=entropy_threshold):
        most_common=df[traget_col].mode()[0]
        return TreeNode(is_leaf=True,prediction=most_common)
    if(len(features)==0):
        most_common=df[target_col].mode()[0]
        return TreeNode(is_leaf=True,prediction=most_common)

    best_feature=None
    best_gain=-1.0

    for feat in features:
        cond_entropy=cal_con_entropy(df,feat,target_col)
        gain=current_entropy-cond_entropy
        if(gain>best_gain):
            best_gain=gain
            best_feature=feat
    
    if(best_gain<=1e-5):
        most_common=df[target_col].mode()[0]
        return TreeNode(is_leaf=True,prediction=most_common)

    root=TreeNode(is_leaf=False,feature=best_feature)
    remaining_features=[f for f in features if f != best_feature]

    feature_values=df[best_feature].unique()
    for val in feature_values:
        sub_df=df[df[best_feature]==val]
        children_node=build_tree(sub_df,remaining_features,target_col,entropy_threshold)
        root.children[val]=children_node
    return root
        
        
    
    

decision_tree=build_tree(df,x_cols,y_col,entropy_threshold=0.1)
decision_tree.display()

🌲 [按特征分裂: texture]
  ├── 当该特征取值为 -> 'soft':
  │   🍃 预测结果: benign
  ├── 当该特征取值为 -> 'firm':
  │   🌲 [按特征分裂: shape]
  │     ├── 当该特征取值为 -> 'lobulated':
  │     │   🍃 预测结果: benign
  │     ├── 当该特征取值为 -> 'round':
  │     │   🍃 预测结果: benign
  │     ├── 当该特征取值为 -> 'irregular':
  │     │   🌲 [按特征分裂: age_group]
  │     │     ├── 当该特征取值为 -> 'old':
  │     │     │   🍃 预测结果: malignant
  │     │     ├── 当该特征取值为 -> 'young':
  │     │     │   🍃 预测结果: malignant
  │     │     ├── 当该特征取值为 -> 'middle':
  │     │     │   🍃 预测结果: benign
  ├── 当该特征取值为 -> 'hard':
  │   🍃 预测结果: malignant
